In [ ]:
# ══════════════════════════════════════════════════════════════════
# Google Colab setup — one cell to rule them all.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
import os, sys

# ① EDIT THIS to point at YOUR fork of the Classroom repo.
#    Your fork URL is at the top of your assignment page on GitHub.
#    Example: "https://github.com/janedoe/pcml-run26-professional-certificate-in-machine-learning-pcml-run26-2601-janedoe.git"
FORK_URL = "https://github.com/<your-github-username>/<your-fork>.git"
REPO_DIR = "/content/pcml-run26"

if not os.path.exists(REPO_DIR):
    !git clone {FORK_URL} {REPO_DIR}

# ② cd into the repo so relative data paths resolve
%cd {REPO_DIR}

# ③ Install deps — Colab has PyTorch preinstalled; add the Kailash stack.
!pip install -q kailash kailash-ml kailash-kaizen kailash-nexus \
               polars plotly gdown python-dotenv nest-asyncio

# ④ Make the `shared` package importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ⑤ Patch asyncio — Colab has a running event loop; nested asyncio.run() fails without this.
import nest_asyncio
nest_asyncio.apply()

# ⑥ GPU check — warn loudly if running on CPU (training will be much slower).
import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU detected — training will be slow.')
    print('  Runtime → Change runtime type → T4 GPU (free tier)')

print("✓ Colab setup complete — shared.mlfp05 is importable")


# ════════════════════════════════════════════════════════════════════════
# MLFP05 Exercise 2.2 — ResNet + Squeeze-and-Excitation: Depth and Attention
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   After completing this file, you will be able to:
#   - Explain WHY deeper networks fail (vanishing gradients, degradation
#     problem) in terms a non-technical manager can understand
#   - Build a residual block (y = F(x) + x) and explain how skip
#     connections keep gradients healthy
#   - Implement Squeeze-and-Excitation (SE) channel attention — "which
#     feature maps matter for THIS specific input?"
#   - Compare ResNetSE against SimpleCNN on the same dataset with
#     ExperimentTracker metrics
#   - Visualise Grad-CAM heatmaps showing WHERE the model looks to make
#     each decision (not just what it predicts, but WHY)
#   - Apply this to manufacturing quality inspection at a Singapore
#     semiconductor fab
#
# PREREQUISITES: M5/ex_2/01_simple_cnn.py (SimpleCNN training, filter
#   visualisation, ExperimentTracker basics)
# ESTIMATED TIME: ~40 min
#
# DATASET: CIFAR-10 — same 50K training images as 01_simple_cnn.py.
#   We re-use the data loading from helpers.py.
#
# PHASES:
#   1. THEORY  — Why depth fails and how residuals + attention fix it
#   2. BUILD   — ResBlock, SEBlock, and ResNetSE architecture
#   3. TRAIN   — Train and compare against SimpleCNN
#   4. VISUALISE — Grad-CAM heatmaps showing model attention
#   5. APPLY   — Semiconductor wafer inspection at GlobalFoundries SG
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
# Copyright 2026 Terrene Foundation
# SPDX-License-Identifier: Apache-2.0
from __future__ import annotations


import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from shared.mlfp05.ex_2 import (
    BATCH_SIZE,
    CLASS_NAMES,
    DEVICE,
    EPOCHS,
    N_CLASSES,
    count_parameters,
    create_visualizer,
    denormalise_cifar,
    init_engines,
    load_cifar10,
    register_model,
    save_training_plots,
    train_model,
)



## PHASE 1 — THEORY: Why Depth Fails, How Residuals Fix It


THE DEGRADATION PROBLEM:
    Deep networks (50+ layers) perform WORSE than shallow ones without
    skip connections. Gradients vanish through the chain of layers.

  RESIDUAL FIX: y = F(x) + x
    The network only learns what to ADD (the residual), not the entire
    transformation. Gradients flow directly through the skip connection.

  SE ATTENTION: "Which feature maps matter for THIS image?"
    Squeeze (global avg pool) -> Excite (small MLP) -> Scale (re-weight)
    Adds <1% parameters, measurable accuracy improvement.


In [ ]:
# THE DEGRADATION PROBLEM (discovered by He et al., 2015):
#
# Intuition for a non-technical audience:
#   Imagine a game of "telephone" with 50 people in a chain. Each
#   person whispers the message to the next. By the time it reaches
#   person #50, the message is garbled beyond recognition.
#
#   That is what happens in a deep neural network without skip
#   connections. Each layer transforms the signal, and tiny errors
#   compound. By layer 50, the gradient (the "correction signal" sent
#   backwards during training) has shrunk to near-zero — the first
#   layers never learn.
#
# THE RESIDUAL FIX:
#   Instead of each person paraphrasing the message, you give person
#   #1 a direct phone line to person #50. Now person #50 can hear
#   BOTH the original message AND the chain's version. If the chain
#   garbled it, the original is still available.
#
#   Mathematically: y = F(x) + x
#   - F(x) is the "chain's version" (what the conv layers compute)
#   - x is the "direct phone line" (the skip connection)
#   - The network only needs to learn the RESIDUAL: F(x) = y - x
#     (what to ADD to the input, not the entire transformation)
#
# WHY THIS MATTERS IN PRACTICE:
#   Before ResNet (2015): networks deeper than ~20 layers performed
#   WORSE than shallower ones, even on training data. Not overfitting —
#   literally unable to optimise.
#
#   After ResNet: 152-layer networks outperformed 20-layer networks.
#   ResNet won the 2015 ImageNet competition with 3.57% top-5 error
#   (human performance: ~5.1%).
#
# SQUEEZE-AND-EXCITATION (SE) ATTENTION:
#   After residual blocks extract features, SE blocks ask: "which of
#   these 32 feature maps are most useful for THIS specific image?"
#
#   Analogy: A doctor examining an X-ray doesn't weigh all visual
#   features equally. For a lung scan, they focus on tissue density
#   patterns. For a bone scan, they focus on edge sharpness. SE blocks
#   learn this "selective attention" automatically.
#
#   Mechanism:
#     1. SQUEEZE: Global average pool each feature map to one number
#        (32 feature maps -> 32 numbers summarising "how active is
#        this filter?")
#     2. EXCITE: A small MLP learns to produce a weight for each
#        channel (32 numbers -> 32 weights between 0 and 1)
#     3. SCALE: Multiply each feature map by its weight (amplify
#        useful channels, suppress irrelevant ones)
#
#   Cost: <1% extra parameters. Benefit: measurable accuracy boost.

print("=" * 70)
print("  PHASE 1 — THEORY: Residual Learning + Channel Attention")
print("=" * 70)
print(
)



## PHASE 2 — BUILD: ResBlock, SEBlock, and ResNetSE


Two conv layers with BatchNorm and ReLU. The skip connection adds
    the input directly to the output, giving gradients a highway through
    the network.


Squeeze: global average pool -> one number per channel
    Excite: small MLP -> one weight per channel (0 to 1 via sigmoid)
    Scale: multiply each feature map by its learned weight


Architecture:
        Stem: Conv(3->32) -> BN -> ReLU -> MaxPool(2)   [32x32 -> 16x16]
        ResBlock(32) -> SEBlock(32) -> ResBlock(32)
        AdaptiveAvgPool -> Linear(32 -> 10)


In [ ]:
print("=" * 70)
print("  PHASE 2 — BUILD: ResBlock + SEBlock + ResNetSE")
print("=" * 70)


class ResBlock(nn.Module):

    def __init__(self, channels: int):
        super().__init__()
        # TODO: Build residual block — two Conv2d(channels, channels, 3, padding=1)
        #   with BatchNorm2d after each conv
        self.conv1 = ____
        self.bn1 = ____
        self.conv2 = ____
        self.bn2 = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Implement residual forward pass
        #   1. Save identity = x
        #   2. out = ReLU(bn1(conv1(x)))
        #   3. out = bn2(conv2(out))
        #   4. return ReLU(out + identity)  <-- the skip connection
        identity = x
        out = ____
        out = ____
        return ____


class SEBlock(nn.Module):

    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        # TODO: Build the SE MLP — nn.Sequential with:
        #   Linear(channels, hidden), ReLU(), Linear(hidden, channels), Sigmoid()
        self.fc = nn.Sequential(
            ____,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, _, _ = x.shape
        # TODO: Implement squeeze-excite-scale
        #   1. Squeeze: adaptive_avg_pool2d(x, 1).view(b, c)  -> one number per channel
        #   2. Excite: self.fc(s).view(b, c, 1, 1)  -> per-channel weights
        #   3. Scale: x * w  -> re-weight each feature map
        s = ____
        w = ____
        return ____


class ResNetSE(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        # TODO: Build the stem — nn.Sequential with Conv2d(3, 32, 3, padding=1),
        #   BatchNorm2d(32), ReLU(), MaxPool2d(2)
        self.stem = nn.Sequential(
            ____,
        )
        # TODO: Wire up ResBlock -> SEBlock -> ResBlock -> pool -> classifier
        self.block1 = ____
        self.se1 = ____
        self.block2 = ____
        self.pool = ____
        self.fc = ____

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Forward pass through stem -> block1 -> se1 -> block2 -> pool -> fc
        x = self.stem(x)
        x = ____
        x = ____
        x = ____
        x = self.pool(x).flatten(1)
        return self.fc(x)



### Checkpoint 1: Architecture builds correctly


In [ ]:
resnet_test = ResNetSE()
dummy_input = torch.randn(2, 3, 32, 32)
dummy_output = resnet_test(dummy_input)
assert dummy_output.shape == (
    2,
    N_CLASSES,
), f"ResNetSE output should be (batch, {N_CLASSES}), got {dummy_output.shape}"

resnet_params = count_parameters(resnet_test)
print(f"\nResNetSE built successfully")
print(f"  Parameters: {resnet_params:,}")
print(f"  Components: stem + ResBlock + SEBlock + ResBlock + classifier")
print(f"  Skip connections: 2 (one per ResBlock)")
print(f"  SE reduction ratio: 8 (32 channels -> 4 hidden -> 32 weights)")

# INTERPRETATION: ResNetSE has more parameters than SimpleCNN due to the
# extra ResBlock convolutions and SE MLP, but the skip connections make
# gradient flow more efficient, so the network trains faster per-parameter.
# The SE block adds only ~200 parameters but can measurably improve accuracy.
print("\n--- Checkpoint 1 passed --- ResNetSE architecture verified\n")
del resnet_test, dummy_input, dummy_output



## PHASE 3 — TRAIN: ResNetSE vs SimpleCNN Comparison


Plain CNN baseline for comparison (same as 01_simple_cnn.py).


In [ ]:
print("=" * 70)
print("  PHASE 3 — TRAIN: ResNetSE on full CIFAR-10")
print("=" * 70)

# Load data
X_train, y_train, X_val, y_val, train_loader, val_loader = load_cifar10()

# Set up engines
conn, tracker, exp_name, registry, has_registry = init_engines()

# TODO: Train ResNetSE
#   1. Instantiate ResNetSE()
#   2. Call train_model(model, "ResNetSE", tracker, exp_name, train_loader, val_loader, epochs=EPOCHS)
print(f"\nTraining ResNetSE for {EPOCHS} epochs on {X_train.shape[0]:,} images...")
resnet_se = ____
resnet_losses, resnet_accs = ____

# Also train SimpleCNN for direct comparison in the same experiment
# Define inline to avoid executing 01_simple_cnn.py as a side effect


class SimpleCNN(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        # TODO: Same SimpleCNN as 01 — Conv(3->32)->BN->ReLU->MaxPool, Conv(32->64)->BN->ReLU->MaxPool
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


print("\nTraining SimpleCNN for comparison...")
simple_cnn_compare = SimpleCNN()
simple_losses, simple_accs = train_model(
    simple_cnn_compare,
    "SimpleCNN_compare",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)



### Checkpoint 2: Training converged


In [ ]:
assert len(resnet_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses"
assert resnet_losses[-1] < resnet_losses[0], "ResNetSE loss should decrease"
assert resnet_accs[-1] > 0.4, (
    f"ResNetSE val accuracy {resnet_accs[-1]:.3f} too low -- "
    "expected > 0.4 on full CIFAR-10"
)

# Architecture comparison table
print(f"\n{'=' * 50}")
print(f"  ARCHITECTURE COMPARISON")
print(f"{'=' * 50}")
print(f"{'Model':>15} {'Params':>10} {'Final Loss':>12} {'Val Acc':>10}")
print("-" * 50)
print(
    f"{'SimpleCNN':>15} {count_parameters(simple_cnn_compare):>10,} "
    f"{simple_losses[-1]:>12.4f} {simple_accs[-1]:>9.3f}"
)
print(
    f"{'ResNetSE':>15} {resnet_params:>10,} "
    f"{resnet_losses[-1]:>12.4f} {resnet_accs[-1]:>9.3f}"
)

improvement = resnet_accs[-1] - simple_accs[-1]
print(
    f"\n  ResNetSE improvement: {improvement:+.3f} ({improvement/simple_accs[-1]:+.1%} relative)"
)

# INTERPRETATION: ResNetSE should achieve higher accuracy than SimpleCNN.
# The skip connections let gradients flow directly through the network,
# and the SE block re-weights channels so the most informative feature
# maps get amplified. The improvement is modest on shallow networks but
# becomes dramatic as depth increases (ResNet-50 vs VGG-19, for example).
print("\n--- Checkpoint 2 passed --- ResNetSE trained and compared\n")

# Register in ModelRegistry
if has_registry:
    # TODO: Register the trained ResNetSE model
    #   register_model(registry, "resnet_se_cifar10", resnet_se, resnet_losses[-1], resnet_accs[-1])
    resnet_version = ____

# Save comparison plots
viz = create_visualizer()
# TODO: Save loss and accuracy comparison plots
#   save_training_plots(viz, {"SimpleCNN loss": ..., "ResNetSE loss": ...}, filename, y_label=...)
____
____



## PHASE 4 — VISUALISE: Grad-CAM Heatmaps


Args:
        model: The CNN model
        input_tensor: Input image (1, C, H, W)
        target_class: Class index to explain
        target_layer: Conv layer to compute CAM for

    Returns:
        Heatmap as numpy array (H, W), values in [0, 1]


In [ ]:
# Grad-CAM (Gradient-weighted Class Activation Mapping) answers:
#   "WHERE in the image did the model look to make its decision?"
#
# It works by:
#   1. Run a forward pass for the target class
#   2. Compute gradients of the target class score w.r.t. the last
#      convolutional layer's feature maps
#   3. Weight each feature map by the average gradient (how important
#      is this feature map for this class?)
#   4. Sum the weighted feature maps and apply ReLU (keep only
#      positive contributions)
#   5. Overlay the heatmap on the original image
#
# This is the gold standard for CNN interpretability. When a medical
# imaging model says "this X-ray shows pneumonia", Grad-CAM shows
# WHETHER the model looked at the lungs (good) or the patient ID label
# on the corner of the film (bad — a known failure mode).

print("=" * 70)
print("  PHASE 4 — VISUALISE: Grad-CAM Heatmaps (Where Does the Model Look?)")
print("=" * 70)


def compute_gradcam(
    model: nn.Module,
    input_tensor: torch.Tensor,
    target_class: int,
    target_layer: nn.Module,
) -> np.ndarray:
    gradients = []
    activations = []

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    def forward_hook(module, input, output):
        activations.append(output.detach())

    bh = target_layer.register_full_backward_hook(backward_hook)
    fh = target_layer.register_forward_hook(forward_hook)

    model.eval()
    model.zero_grad()

    # TODO: Forward + backward pass for Grad-CAM
    #   1. output = model(input_tensor)
    #   2. target_score = output[0, target_class]
    #   3. target_score.backward()
    output = ____
    target_score = ____
    target_score.backward()

    bh.remove()
    fh.remove()

    # TODO: Compute the Grad-CAM heatmap
    #   1. grads = gradients[0].squeeze(0)  — shape (C, H, W)
    #   2. acts = activations[0].squeeze(0)  — shape (C, H, W)
    #   3. weights = grads.mean(dim=(1, 2))  — average gradient per channel
    #   4. cam = (weights.unsqueeze(1).unsqueeze(2) * acts).sum(dim=0)  — weighted sum
    #   5. cam = F.relu(cam)  — only positive contributions
    #   6. Normalise to [0, 1]: if cam.max() > 0: cam = cam / cam.max()
    #   7. Upsample: F.interpolate(cam[None, None], size=(H, W), mode="bilinear", align_corners=False).squeeze()
    grads = ____
    acts = ____
    weights = ____
    cam = ____
    cam = F.relu(cam)

    if cam.max() > 0:
        cam = cam / cam.max()

    cam = F.interpolate(
        cam.unsqueeze(0).unsqueeze(0),
        size=(input_tensor.shape[2], input_tensor.shape[3]),
        mode="bilinear",
        align_corners=False,
    ).squeeze()

    return cam.numpy()


# Generate Grad-CAM for several validation images
sample_indices = [0, 42, 100, 500, 1000, 2000, 3000, 5000]

# Use the last conv layer in the second ResBlock
target_layer = resnet_se.block2.conv2

fig_gradcam, axes = plt.subplots(3, len(sample_indices), figsize=(24, 10))
fig_gradcam.suptitle(
    "Grad-CAM: Where Does ResNetSE Look? (Red = High Attention)",
    fontsize=14,
)

for col, idx in enumerate(sample_indices):
    img = X_val[idx : idx + 1].clone().requires_grad_(True)
    true_label = y_val[idx].item()
    true_name = CLASS_NAMES[true_label]

    # Get prediction
    with torch.no_grad():
        logits = resnet_se(X_val[idx : idx + 1])
        pred_label = logits.argmax(dim=-1).item()
        pred_name = CLASS_NAMES[pred_label]
        conf = F.softmax(logits, dim=-1)[0, pred_label].item()

    # TODO: Compute Grad-CAM for predicted class
    #   heatmap = compute_gradcam(resnet_se, img, pred_label, target_layer)
    heatmap = ____

    # Row 0: Original image
    orig = denormalise_cifar(X_val[idx])
    axes[0, col].imshow(orig.permute(1, 2, 0).numpy())
    correct = pred_label == true_label
    colour = "green" if correct else "red"
    axes[0, col].set_title(f"True: {true_name}", fontsize=8)
    axes[0, col].axis("off")

    # Row 1: Grad-CAM heatmap
    axes[1, col].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
    axes[1, col].set_title(f"Pred: {pred_name} ({conf:.0%})", fontsize=8, color=colour)
    axes[1, col].axis("off")

    # Row 2: Overlay (heatmap on original)
    orig_np = orig.permute(1, 2, 0).numpy()
    heatmap_colour = plt.cm.jet(heatmap)[:, :, :3]
    overlay = 0.5 * orig_np + 0.5 * heatmap_colour
    overlay = np.clip(overlay, 0, 1)
    axes[2, col].imshow(overlay)
    axes[2, col].set_title("Overlay", fontsize=8)
    axes[2, col].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Grad-CAM", fontsize=10)
axes[2, 0].set_ylabel("Overlay", fontsize=10)

plt.tight_layout()
plt.savefig("ex_2_02_gradcam.png", dpi=150, bbox_inches="tight")
plt.close(fig_gradcam)
print("  Saved: ex_2_02_gradcam.png")
print("  Red regions = where the model focused to make its prediction")
print("  Check: does the model look at the object or the background?")

# SE attention weight analysis
print("\n  SE ATTENTION WEIGHT ANALYSIS:")
resnet_se.eval()
se_weights_per_class: dict[str, list[np.ndarray]] = {name: [] for name in CLASS_NAMES}

with torch.no_grad():
    se_outputs = []

    def se_hook(module, input, output):
        b, c, _, _ = input[0].shape
        s = F.adaptive_avg_pool2d(input[0], 1).view(b, c)
        w = module.fc(s)
        se_outputs.append(w.cpu().numpy())

    hook_handle = resnet_se.se1.register_forward_hook(se_hook)

    for batch_start in range(0, min(2000, len(X_val)), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, 2000)
        _ = resnet_se(X_val[batch_start:batch_end])

    hook_handle.remove()

    all_se_weights = np.concatenate(se_outputs, axis=0)
    for i in range(min(2000, len(y_val))):
        label = CLASS_NAMES[y_val[i].item()]
        se_weights_per_class[label].append(all_se_weights[i])

# Show average SE weights per class
print(f"  {'Class':>12s}  Top-3 channels (most attended)")
print("  " + "-" * 50)
for cls_name in CLASS_NAMES:
    if se_weights_per_class[cls_name]:
        avg_weights = np.mean(se_weights_per_class[cls_name], axis=0)
        top3 = np.argsort(avg_weights)[-3:][::-1]
        top3_str = ", ".join(f"ch{c}({avg_weights[c]:.2f})" for c in top3)
        print(f"  {cls_name:>12s}  {top3_str}")



### Checkpoint 3: Visualisations generated


In [ ]:
import os

assert os.path.exists("ex_2_02_gradcam.png"), "Grad-CAM visualisation missing"
assert os.path.exists("ex_2_02_arch_comparison_loss.html"), "Loss comparison missing"
print("\n--- Checkpoint 3 passed --- Grad-CAM and SE analysis complete\n")



## PHASE 5 — APPLY: Semiconductor Wafer Inspection at GlobalFoundries SG


WAFER INSPECTION SIMULATION ({total:,} inspection images):

  Detection Performance:
    Defects correctly caught:  {caught:>5,} / {actual_defects:,} ({detection_rate:.1%} detection rate)
    Defects missed (CRITICAL): {missed:>5,} / {actual_defects:,} ({1-detection_rate:.1%} miss rate)
    False alarms:              {false_alarm:>5,} / {actual_passes:,} ({false_alarm_rate:.1%} false alarm rate)
    Correct passes:            {correct_pass:>5,} / {actual_passes:,}

  Financial Impact (scaled to GlobalFoundries production volume):
    Monthly wafer throughput:         ~10,000 wafers
    Monthly inspection images:        ~200,000

    COST OF MISSED DEFECTS (this simulation):
      Missed defects: {missed:,} at avg ${inspection_results["missed_cost"]/max(missed,1):,.0f}/defect
      Total missed defect cost: ${inspection_results["missed_cost"]:>12,.0f}

    COST COMPARISON (monthly, at production scale):
      Human inspection (15 inspectors x 3 shifts):  $  202,500
      CNN system (compute + 5 edge-case inspectors): $   80,000
      CNN-missed defect scrap (estimated):           $   45,000
      Net monthly savings:                           $   77,500
      Projected annual savings:                      $  930,000

  WHY GRAD-CAM IS CRITICAL HERE:
    When the CNN flags a defect, the inspector needs to know WHERE.
    Without Grad-CAM: inspector re-scans entire 512x512 image (~30 sec)
    With Grad-CAM: inspector zooms to highlighted region (~5 sec)
    Time savings: 83% faster verification per flagged image

  STAKEHOLDER-READY OUTPUT:
    "The CNN inspection system detects {detection_rate:.0%} of wafer defects
    automatically, with a {false_alarm_rate:.0%} false alarm rate.
    Grad-CAM heatmaps show inspectors exactly where each defect is,
    reducing verification time from 30 seconds to 5 seconds.
    Projected annual savings: $930K from reduced headcount + $2-5M
    from catching fatigue-related misses in late-shift inspection."


In [ ]:
# SCENARIO: You are an ML engineer at GlobalFoundries' Singapore fab
# (Woodlands). The fab produces 300mm wafers for automotive, IoT, and
# 5G chips. Each wafer goes through 500+ processing steps over 3 months.
# Defects at ANY step can scrap the entire wafer ($5,000-$50,000 each).
#
# CURRENT PROCESS:
#   - Automated Optical Inspection (AOI) machines capture high-res
#     images at 20+ inspection points per wafer
#   - Each image is reviewed by a human inspector who classifies:
#     PASS, PARTICLE, SCRATCH, PATTERN_DEFECT, CONTAMINATION
#   - 15 inspectors per shift, 3 shifts/day, ~2,000 images/inspector/day
#   - Human accuracy: ~92% (fatigue causes misses in late-shift hours)
#   - False negative cost: $5,000-$50,000 (defective wafer continues
#     through remaining processing steps, wasting all downstream work)
#   - False positive cost: $200-$500 (unnecessary re-inspection)
#
# WHY RESNET + SE + GRAD-CAM:
#   1. ResNet handles high-resolution inspection images (512x512+)
#      without the degradation problem
#   2. SE attention learns which feature channels are important for
#      each defect type (scratches activate edge channels; particles
#      activate texture channels; contamination activates colour channels)
#   3. Grad-CAM provides EXPLAINABILITY: when the model flags a defect,
#      it shows WHERE on the wafer the defect was detected. Inspectors
#      can verify in seconds instead of re-scanning the entire image.
#
# BUSINESS CASE:
#   Manual inspection: 15 inspectors * 3 shifts * $4,500/month = $202,500/month
#   CNN system: $30,000/month compute + $50,000/month (5 inspectors for edge cases)
#   Monthly savings: $122,500
#   Annual savings: $1.47M
#   Additional: CNN catches late-shift fatigue misses -> estimated $2-5M
#   in avoided scrap per year

print("=" * 70)
print("  PHASE 5 — APPLY: Semiconductor Wafer Inspection (GlobalFoundries SG)")
print("=" * 70)

# TODO: Build the wafer inspection simulation
#   1. Create WAFER_MAPPING: airplane/ship -> "PASS", automobile/truck -> "PATTERN_DEFECT",
#      bird/horse -> "PARTICLE", cat/dog -> "SCRATCH", deer/frog -> "CONTAMINATION"
#   2. Create DEFECT_COST: PASS=0, PARTICLE=15000, SCRATCH=25000,
#      PATTERN_DEFECT=8000, CONTAMINATION=35000
#   3. Run inference and classify results
WAFER_MAPPING = {____}

DEFECT_COST = {____}

resnet_se.eval()
with torch.no_grad():
    val_logits = resnet_se(X_val)
    val_probs = F.softmax(val_logits, dim=-1)
    val_preds = val_logits.argmax(dim=-1)
    val_confidences = val_probs.gather(1, val_preds.unsqueeze(1)).squeeze()

# TODO: Classify each image as wafer inspection result
#   Loop through all validation images, map CIFAR predictions to wafer categories,
#   count: correct_defect_caught, missed_defect, false_alarm, correct_pass
#   Track missed_cost and false_alarm_cost
inspection_results = {
    "total": 0,
    "correct_defect_caught": 0,
    "missed_defect": 0,
    "false_alarm": 0,
    "correct_pass": 0,
    "missed_cost": 0.0,
    "false_alarm_cost": 0.0,
}

INSPECTION_THRESHOLD = 0.70

for i in range(len(y_val)):
    true_cifar = CLASS_NAMES[y_val[i].item()]
    pred_cifar = CLASS_NAMES[val_preds[i].item()]
    true_wafer = WAFER_MAPPING[true_cifar]
    pred_wafer = WAFER_MAPPING[pred_cifar]

    inspection_results["total"] += 1

    # TODO: Implement the classification logic
    #   if true is defect AND pred is defect -> correct_defect_caught
    #   if true is defect AND pred is PASS -> missed_defect (add DEFECT_COST[true_wafer])
    #   if true is PASS AND pred is defect -> false_alarm (add $300 re-inspection)
    #   else -> correct_pass
    ____

total = inspection_results["total"]
caught = inspection_results["correct_defect_caught"]
missed = inspection_results["missed_defect"]
false_alarm = inspection_results["false_alarm"]
correct_pass = inspection_results["correct_pass"]

actual_defects = caught + missed
actual_passes = correct_pass + false_alarm
detection_rate = caught / actual_defects if actual_defects > 0 else 0
false_alarm_rate = false_alarm / actual_passes if actual_passes > 0 else 0

print(
)



### Checkpoint 4: Apply section complete


In [ ]:
assert inspection_results["total"] == len(y_val), "Should inspect all samples"
assert (
    detection_rate > 0.3
), f"Detection rate {detection_rate:.3f} too low -- model should catch some defects"
print("--- Checkpoint 4 passed --- wafer inspection application demonstrated\n")



## Clean up


In [ ]:
asyncio.run(conn.close())



## REFLECTION


THEORY:
  [x] The degradation problem: deeper is not always better without
      skip connections (the "telephone game" problem)
  [x] Residual learning: y = F(x) + x -- learn the residual, not the
      full transformation. Gradients flow directly through shortcuts.
  [x] SE attention: squeeze (pool) -> excite (MLP) -> scale (re-weight)
      Learns which channels matter for each specific input.

  BUILD + TRAIN:
  [x] ResBlock: two-conv residual block with BatchNorm
  [x] SEBlock: channel attention with <1% parameter overhead
  [x] ResNetSE: {{resnet_params:,}} params, {{resnet_accs[-1]:.1%}} val accuracy
  [x] Compared against SimpleCNN: {{improvement:+.3f}} accuracy improvement

  VISUALISE (the proof):
  [x] Grad-CAM heatmaps: WHERE the model looks for each prediction
  [x] SE attention weights: WHICH channels each class relies on
  [x] Overlay visualisation: heatmap on original for instant verification

  APPLY:
  [x] GlobalFoundries Singapore semiconductor wafer inspection
  [x] Detection rate: {{detection_rate:.0%}} of defects caught automatically
  [x] Grad-CAM enables 83% faster inspector verification (30s -> 5s)
  [x] Projected annual savings: $930K + $2-5M avoided scrap

  KEY INSIGHT: Residual connections and SE attention are not academic
  curiosities -- they are production requirements. ResNets enabled the
  first superhuman image classifiers. SE attention costs almost nothing
  but tells you WHICH features the model cares about. And Grad-CAM
  turns a "black box" classifier into an explainable decision that
  inspectors, regulators, and stakeholders can verify visually.

  Next: In 03_production_pipeline.py, you'll export the best model to
  ONNX and serve it through InferenceServer -- the bridge from
  "it works in a notebook" to "it works in production"...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)



## DIAGNOSTIC CHECKPOINT — five instruments (residual connections)


In [ ]:
# ResNetSE = residual blocks + squeeze-excitation. Residuals are
# the PROVEN fix for vanishing gradients (compare to ex_1/07's
# 5-layer stacked AE). Expect near-uniform gradient RMS across
# depth — that is the whole point of skip connections.
from shared.mlfp05.diagnostics import diagnose_classifier

print("\n── Diagnostic Report (ResNetSE) ──")
diag, findings = diagnose_classifier(
    resnet_se,
    val_loader,
    title="ResNetSE (CIFAR-10)",
    n_batches=8,
    train_losses=resnet_losses,
    val_losses=[1.0 - a for a in resnet_accs],
    show=False,
)
# ══════ EXPECTED OUTPUT (synthesized reference — full run produces similar pattern) ══════



## DL Diagnostics Report — Prescription Pad


In [ ]:
#   [✓] Gradient flow (HEALTHY): min RMS = 6.2e-04 at
#       'layer3.1.conv2.weight' (deepest block). Spread across
#       16 Conv layers = 8.3x — nearly uniform. Skip
#       connections are doing their job.
#   [✓] Dead neurons  (HEALTHY): max 4% dead on layer1.0 —
#       well below 15% flag. SE blocks' channel re-weighting
#       keeps every filter engaged.
#   [✓] Loss trend    (HEALTHY): train slope -4.8e-02/epoch,
#       val slope -3.9e-02/epoch. Train-val gap 6% at final
#       epoch — no overfitting thanks to augmentation.



## Final val acc: ~0.62 after 8 epochs on CIFAR-10.

STUDENT INTERPRETATION GUIDE — reading the Prescription Pad:

 [BLOOD TEST — RESIDUAL CONNECTIONS AT WORK] Gradient spread
    8.3x across 16 layers is the RESNET SUCCESS SIGNATURE.
    Contrast ex_1/07 stacked AE (5 dense layers → 750x
    spread). He et al. 2016 (Slide 5P) showed additive skip
    connections (y = F(x) + x) let gradients flow unchanged
    from the loss back to any depth — the "gradient
    highway". Even a 50-layer ResNet trains stably.
    >> Prescription: If RMS spread exceeds 100x across the
       network, a skip connection is mis-wired. Check the
       addition dimension: F(x) must have shape IDENTICAL
       to x (or projection-adapted via 1x1 conv). Mismatch
       silently breaks the residual path.

 [X-RAY — SE BLOCK CONTRIBUTION] 4% dead max is lower than
    a plain ResNet (typically 8-12% at this depth). The
    Squeeze-and-Excitation blocks (Hu et al. 2018)
    re-weight channels per sample, so even a channel that
    would be dead under one input gets promoted under
    another. This is the architectural answer to ReLU
    saturation without sacrificing the non-linearity.
    >> Prescription: If dead% exceeds 10%, your SE
       reduction ratio is too aggressive. Change
       reduction from 16 to 8 so the squeeze bottleneck
       preserves more channel-specific signal.

 [STETHOSCOPE — NO OVERFITTING] Train-val gap 6% means
    augmentation (flip + crop) is delivering regularisation
    WITHOUT underfitting. If gap >15%, reduce augmentation
    strength (smaller padding, less colour jitter). If gap
    <2%, augmentation is TOO aggressive — model isn't
    learning the core distribution.
    >> Prescription: Target 5-10% gap on CIFAR-10 at 8
       epochs. Stronger augmentation (cutout, mixup) for
       longer training runs.

 FIVE-INSTRUMENT TAKEAWAY: ResNet-SE is the "everything
 healthy" reference for deep nets. Same Blood Test +
 X-Ray metrics you've used since ex_1/01, but now all
 green because the architecture MATCHES the data. This
 sets the bar for 03_production_pipeline (must stay
 healthy pre-export) and 04_hyperparameter_study (which
 HP configs break which instruments?).


Plain CNN baseline for comparison (same as 01_simple_cnn.py).


In [ ]:
# Also train SimpleCNN for direct comparison in the same experiment
# Define inline to avoid executing 01_simple_cnn.py as a side effect


class SimpleCNN(nn.Module):

    def __init__(self, n_classes: int = N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.features(x))


print("\nTraining SimpleCNN for comparison...")
simple_cnn_compare = SimpleCNN()
simple_losses, simple_accs = train_model(
    simple_cnn_compare,
    "SimpleCNN_compare",
    tracker,
    exp_name,
    train_loader,
    val_loader,
    epochs=EPOCHS,
)



### Checkpoint 2: Training converged


In [ ]:
assert len(resnet_losses) == EPOCHS, f"Expected {EPOCHS} epoch losses"
assert resnet_losses[-1] < resnet_losses[0], "ResNetSE loss should decrease"
assert resnet_accs[-1] > 0.4, (
    f"ResNetSE val accuracy {resnet_accs[-1]:.3f} too low -- "
    "expected > 0.4 on full CIFAR-10"
)

# Architecture comparison table
print(f"\n{'=' * 50}")
print(f"  ARCHITECTURE COMPARISON")
print(f"{'=' * 50}")
print(f"{'Model':>15} {'Params':>10} {'Final Loss':>12} {'Val Acc':>10}")
print("-" * 50)
print(
    f"{'SimpleCNN':>15} {count_parameters(simple_cnn_compare):>10,} "
    f"{simple_losses[-1]:>12.4f} {simple_accs[-1]:>9.3f}"
)
print(
    f"{'ResNetSE':>15} {resnet_params:>10,} "
    f"{resnet_losses[-1]:>12.4f} {resnet_accs[-1]:>9.3f}"
)

improvement = resnet_accs[-1] - simple_accs[-1]
print(
    f"\n  ResNetSE improvement: {improvement:+.3f} ({improvement/simple_accs[-1]:+.1%} relative)"
)

# INTERPRETATION: ResNetSE should achieve higher accuracy than SimpleCNN.
# The skip connections let gradients flow directly through the network,
# and the SE block re-weights channels so the most informative feature
# maps get amplified. The improvement is modest on shallow networks but
# becomes dramatic as depth increases (ResNet-50 vs VGG-19, for example).
print("\n--- Checkpoint 2 passed --- ResNetSE trained and compared\n")

# Register in ModelRegistry
if has_registry:
    resnet_version = register_model(
        registry,
        "resnet_se_cifar10",
        resnet_se,
        resnet_losses[-1],
        resnet_accs[-1],
    )

# Save comparison plots
viz = create_visualizer()
save_training_plots(
    viz,
    {"SimpleCNN loss": simple_losses, "ResNetSE loss": resnet_losses},
    "ex_2_02_arch_comparison_loss.html",
    y_label="Training Loss",
)
save_training_plots(
    viz,
    {"SimpleCNN accuracy": simple_accs, "ResNetSE accuracy": resnet_accs},
    "ex_2_02_arch_comparison_acc.html",
    y_label="Validation Accuracy",
)



## PHASE 4 — VISUALISE: Grad-CAM Heatmaps


Args:
        model: The CNN model
        input_tensor: Input image (1, C, H, W)
        target_class: Class index to explain
        target_layer: Conv layer to compute CAM for

    Returns:
        Heatmap as numpy array (H, W), values in [0, 1]


In [ ]:
# Grad-CAM (Gradient-weighted Class Activation Mapping) answers:
#   "WHERE in the image did the model look to make its decision?"
#
# It works by:
#   1. Run a forward pass for the target class
#   2. Compute gradients of the target class score w.r.t. the last
#      convolutional layer's feature maps
#   3. Weight each feature map by the average gradient (how important
#      is this feature map for this class?)
#   4. Sum the weighted feature maps and apply ReLU (keep only
#      positive contributions)
#   5. Overlay the heatmap on the original image
#
# This is the gold standard for CNN interpretability. When a medical
# imaging model says "this X-ray shows pneumonia", Grad-CAM shows
# WHETHER the model looked at the lungs (good) or the patient ID label
# on the corner of the film (bad — a known failure mode).

print("=" * 70)
print("  PHASE 4 — VISUALISE: Grad-CAM Heatmaps (Where Does the Model Look?)")
print("=" * 70)


def compute_gradcam(
    model: nn.Module,
    input_tensor: torch.Tensor,
    target_class: int,
    target_layer: nn.Module,
) -> np.ndarray:
    gradients = []
    activations = []

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0].detach())

    def forward_hook(module, input, output):
        activations.append(output.detach())

    bh = target_layer.register_full_backward_hook(backward_hook)
    fh = target_layer.register_forward_hook(forward_hook)

    model.eval()
    model.zero_grad()

    # Forward pass
    output = model(input_tensor)
    # Backward pass for target class
    target_score = output[0, target_class]
    target_score.backward()

    bh.remove()
    fh.remove()

    # Grad-CAM computation
    grads = gradients[0].squeeze(0)  # (C, H, W)
    acts = activations[0].squeeze(0)  # (C, H, W)

    # Average gradient per channel (importance weight)
    weights = grads.mean(dim=(1, 2))  # (C,)

    # Weighted sum of activations
    cam = (weights.unsqueeze(1).unsqueeze(2) * acts).sum(dim=0)  # (H, W)
    cam = F.relu(cam)  # Only positive contributions

    # Normalise to [0, 1]
    if cam.max() > 0:
        cam = cam / cam.max()

    # Upsample to input resolution
    cam = F.interpolate(
        cam.unsqueeze(0).unsqueeze(0),
        size=(input_tensor.shape[2], input_tensor.shape[3]),
        mode="bilinear",
        align_corners=False,
    ).squeeze()

    return cam.numpy()


# Generate Grad-CAM for several validation images
sample_indices = [0, 42, 100, 500, 1000, 2000, 3000, 5000]

# Use the last conv layer in the second ResBlock
target_layer = resnet_se.block2.conv2

fig_gradcam, axes = plt.subplots(3, len(sample_indices), figsize=(24, 10))
fig_gradcam.suptitle(
    "Grad-CAM: Where Does ResNetSE Look? (Red = High Attention)",
    fontsize=14,
)

for col, idx in enumerate(sample_indices):
    img = X_val[idx : idx + 1].clone().requires_grad_(True)
    true_label = y_val[idx].item()
    true_name = CLASS_NAMES[true_label]

    # Get prediction
    with torch.no_grad():
        logits = resnet_se(X_val[idx : idx + 1])
        pred_label = logits.argmax(dim=-1).item()
        pred_name = CLASS_NAMES[pred_label]
        conf = F.softmax(logits, dim=-1)[0, pred_label].item()

    # Compute Grad-CAM for predicted class
    heatmap = compute_gradcam(resnet_se, img, pred_label, target_layer)

    # Row 0: Original image
    orig = denormalise_cifar(X_val[idx])
    axes[0, col].imshow(orig.permute(1, 2, 0).numpy())
    correct = pred_label == true_label
    colour = "green" if correct else "red"
    axes[0, col].set_title(f"True: {true_name}", fontsize=8)
    axes[0, col].axis("off")

    # Row 1: Grad-CAM heatmap
    axes[1, col].imshow(heatmap, cmap="jet", vmin=0, vmax=1)
    axes[1, col].set_title(f"Pred: {pred_name} ({conf:.0%})", fontsize=8, color=colour)
    axes[1, col].axis("off")

    # Row 2: Overlay (heatmap on original)
    orig_np = orig.permute(1, 2, 0).numpy()
    heatmap_colour = plt.cm.jet(heatmap)[:, :, :3]
    overlay = 0.5 * orig_np + 0.5 * heatmap_colour
    overlay = np.clip(overlay, 0, 1)
    axes[2, col].imshow(overlay)
    axes[2, col].set_title("Overlay", fontsize=8)
    axes[2, col].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Grad-CAM", fontsize=10)
axes[2, 0].set_ylabel("Overlay", fontsize=10)

plt.tight_layout()
plt.savefig("ex_2_02_gradcam.png", dpi=150, bbox_inches="tight")
plt.close(fig_gradcam)
print("  Saved: ex_2_02_gradcam.png")
print("  Red regions = where the model focused to make its prediction")
print("  Check: does the model look at the object or the background?")

# SE attention weight analysis
print("\n  SE ATTENTION WEIGHT ANALYSIS:")
resnet_se.eval()
se_weights_per_class: dict[str, list[np.ndarray]] = {name: [] for name in CLASS_NAMES}

with torch.no_grad():
    # Capture SE weights for a batch of images
    se_outputs = []

    def se_hook(module, input, output):
        # The SE block outputs x * w; we want w
        b, c, _, _ = input[0].shape
        s = F.adaptive_avg_pool2d(input[0], 1).view(b, c)
        w = module.fc(s)
        se_outputs.append(w.cpu().numpy())

    hook_handle = resnet_se.se1.register_forward_hook(se_hook)

    for batch_start in range(0, min(2000, len(X_val)), BATCH_SIZE):
        batch_end = min(batch_start + BATCH_SIZE, 2000)
        _ = resnet_se(X_val[batch_start:batch_end])

    hook_handle.remove()

    all_se_weights = np.concatenate(se_outputs, axis=0)  # (N, 32)
    for i in range(min(2000, len(y_val))):
        label = CLASS_NAMES[y_val[i].item()]
        se_weights_per_class[label].append(all_se_weights[i])

# Show average SE weights per class (which channels each class relies on)
print(f"  {'Class':>12s}  Top-3 channels (most attended)")
print("  " + "-" * 50)
for cls_name in CLASS_NAMES:
    if se_weights_per_class[cls_name]:
        avg_weights = np.mean(se_weights_per_class[cls_name], axis=0)
        top3 = np.argsort(avg_weights)[-3:][::-1]
        top3_str = ", ".join(f"ch{c}({avg_weights[c]:.2f})" for c in top3)
        print(f"  {cls_name:>12s}  {top3_str}")



### Checkpoint 3: Visualisations generated


In [ ]:
import os

assert os.path.exists("ex_2_02_gradcam.png"), "Grad-CAM visualisation missing"
assert os.path.exists("ex_2_02_arch_comparison_loss.html"), "Loss comparison missing"
print("\n--- Checkpoint 3 passed --- Grad-CAM and SE analysis complete\n")



## PHASE 5 — APPLY: Semiconductor Wafer Inspection at GlobalFoundries SG


WAFER INSPECTION SIMULATION ({total:,} inspection images):

  Detection Performance:
    Defects correctly caught:  {caught:>5,} / {actual_defects:,} ({detection_rate:.1%} detection rate)
    Defects missed (CRITICAL): {missed:>5,} / {actual_defects:,} ({1-detection_rate:.1%} miss rate)
    False alarms:              {false_alarm:>5,} / {actual_passes:,} ({false_alarm_rate:.1%} false alarm rate)
    Correct passes:            {correct_pass:>5,} / {actual_passes:,}

  Financial Impact (scaled to GlobalFoundries production volume):
    Monthly wafer throughput:         ~10,000 wafers
    Monthly inspection images:        ~200,000

    COST OF MISSED DEFECTS (this simulation):
      Missed defects: {missed:,} at avg ${inspection_results["missed_cost"]/max(missed,1):,.0f}/defect
      Total missed defect cost: ${inspection_results["missed_cost"]:>12,.0f}

    COST COMPARISON (monthly, at production scale):
      Human inspection (15 inspectors x 3 shifts):  $  202,500
      CNN system (compute + 5 edge-case inspectors): $   80,000
      CNN-missed defect scrap (estimated):           $   45,000
      Net monthly savings:                           $   77,500
      Projected annual savings:                      $  930,000

  WHY GRAD-CAM IS CRITICAL HERE:
    When the CNN flags a defect, the inspector needs to know WHERE.
    Without Grad-CAM: inspector re-scans entire 512x512 image (~30 sec)
    With Grad-CAM: inspector zooms to highlighted region (~5 sec)
    Time savings: 83% faster verification per flagged image

  STAKEHOLDER-READY OUTPUT:
    "The CNN inspection system detects {detection_rate:.0%} of wafer defects
    automatically, with a {false_alarm_rate:.0%} false alarm rate.
    Grad-CAM heatmaps show inspectors exactly where each defect is,
    reducing verification time from 30 seconds to 5 seconds.
    Projected annual savings: $930K from reduced headcount + $2-5M
    from catching fatigue-related misses in late-shift inspection."


In [ ]:
# SCENARIO: You are an ML engineer at GlobalFoundries' Singapore fab
# (Woodlands). The fab produces 300mm wafers for automotive, IoT, and
# 5G chips. Each wafer goes through 500+ processing steps over 3 months.
# Defects at ANY step can scrap the entire wafer ($5,000-$50,000 each).
#
# CURRENT PROCESS:
#   - Automated Optical Inspection (AOI) machines capture high-res
#     images at 20+ inspection points per wafer
#   - Each image is reviewed by a human inspector who classifies:
#     PASS, PARTICLE, SCRATCH, PATTERN_DEFECT, CONTAMINATION
#   - 15 inspectors per shift, 3 shifts/day, ~2,000 images/inspector/day
#   - Human accuracy: ~92% (fatigue causes misses in late-shift hours)
#   - False negative cost: $5,000-$50,000 (defective wafer continues
#     through remaining processing steps, wasting all downstream work)
#   - False positive cost: $200-$500 (unnecessary re-inspection)
#
# WHY RESNET + SE + GRAD-CAM:
#   1. ResNet handles high-resolution inspection images (512x512+)
#      without the degradation problem
#   2. SE attention learns which feature channels are important for
#      each defect type (scratches activate edge channels; particles
#      activate texture channels; contamination activates colour channels)
#   3. Grad-CAM provides EXPLAINABILITY: when the model flags a defect,
#      it shows WHERE on the wafer the defect was detected. Inspectors
#      can verify in seconds instead of re-scanning the entire image.
#
# BUSINESS CASE:
#   Manual inspection: 15 inspectors * 3 shifts * $4,500/month = $202,500/month
#   CNN system: $30,000/month compute + $50,000/month (5 inspectors for edge cases)
#   Monthly savings: $122,500
#   Annual savings: $1.47M
#   Additional: CNN catches late-shift fatigue misses -> estimated $2-5M
#   in avoided scrap per year

print("=" * 70)
print("  PHASE 5 — APPLY: Semiconductor Wafer Inspection (GlobalFoundries SG)")
print("=" * 70)

# Simulate wafer inspection using CIFAR-10 as proxy
# Map: airplane/ship = PASS (uniform surfaces)
#       cat/dog/deer/horse/bird/frog = defect patterns (complex textures)
#       automobile/truck = PATTERN_DEFECT (regular geometric structures)
WAFER_MAPPING = {
    "airplane": "PASS",
    "ship": "PASS",
    "automobile": "PATTERN_DEFECT",
    "truck": "PATTERN_DEFECT",
    "bird": "PARTICLE",
    "cat": "SCRATCH",
    "deer": "CONTAMINATION",
    "dog": "SCRATCH",
    "frog": "CONTAMINATION",
    "horse": "PARTICLE",
}

DEFECT_COST = {
    "PASS": 0,
    "PARTICLE": 15000,
    "SCRATCH": 25000,
    "PATTERN_DEFECT": 8000,
    "CONTAMINATION": 35000,
}

resnet_se.eval()
with torch.no_grad():
    val_logits = resnet_se(X_val)
    val_probs = F.softmax(val_logits, dim=-1)
    val_preds = val_logits.argmax(dim=-1)
    val_confidences = val_probs.gather(1, val_preds.unsqueeze(1)).squeeze()

# Classify each validation image as wafer inspection result
inspection_results = {
    "total": 0,
    "correct_defect_caught": 0,
    "missed_defect": 0,
    "false_alarm": 0,
    "correct_pass": 0,
    "missed_cost": 0.0,
    "false_alarm_cost": 0.0,
}

INSPECTION_THRESHOLD = 0.70  # confidence threshold for auto-classification

for i in range(len(y_val)):
    true_cifar = CLASS_NAMES[y_val[i].item()]
    pred_cifar = CLASS_NAMES[val_preds[i].item()]
    true_wafer = WAFER_MAPPING[true_cifar]
    pred_wafer = WAFER_MAPPING[pred_cifar]
    conf = val_confidences[i].item()

    inspection_results["total"] += 1

    if true_wafer != "PASS" and pred_wafer != "PASS":
        inspection_results["correct_defect_caught"] += 1
    elif true_wafer != "PASS" and pred_wafer == "PASS":
        inspection_results["missed_defect"] += 1
        inspection_results["missed_cost"] += DEFECT_COST[true_wafer]
    elif true_wafer == "PASS" and pred_wafer != "PASS":
        inspection_results["false_alarm"] += 1
        inspection_results["false_alarm_cost"] += 300  # re-inspection cost
    else:
        inspection_results["correct_pass"] += 1

total = inspection_results["total"]
caught = inspection_results["correct_defect_caught"]
missed = inspection_results["missed_defect"]
false_alarm = inspection_results["false_alarm"]
correct_pass = inspection_results["correct_pass"]

# Calculate rates (among actual defects and actual passes)
actual_defects = caught + missed
actual_passes = correct_pass + false_alarm
detection_rate = caught / actual_defects if actual_defects > 0 else 0
false_alarm_rate = false_alarm / actual_passes if actual_passes > 0 else 0

print(
)



### Checkpoint 4: Apply section complete


In [ ]:
assert inspection_results["total"] == len(y_val), "Should inspect all samples"
assert (
    detection_rate > 0.3
), f"Detection rate {detection_rate:.3f} too low -- model should catch some defects"
print("--- Checkpoint 4 passed --- wafer inspection application demonstrated\n")



## Clean up


In [ ]:
asyncio.run(conn.close())



## REFLECTION


THEORY:
  [x] The degradation problem: deeper is not always better without
      skip connections (the "telephone game" problem)
  [x] Residual learning: y = F(x) + x -- learn the residual, not the
      full transformation. Gradients flow directly through shortcuts.
  [x] SE attention: squeeze (pool) -> excite (MLP) -> scale (re-weight)
      Learns which channels matter for each specific input.

  BUILD + TRAIN:
  [x] ResBlock: two-conv residual block with BatchNorm
  [x] SEBlock: channel attention with <1% parameter overhead
  [x] ResNetSE: {resnet_params:,} params, {resnet_accs[-1]:.1%} val accuracy
  [x] Compared against SimpleCNN: {improvement:+.3f} accuracy improvement

  VISUALISE (the proof):
  [x] Grad-CAM heatmaps: WHERE the model looks for each prediction
  [x] SE attention weights: WHICH channels each class relies on
  [x] Overlay visualisation: heatmap on original for instant verification

  APPLY:
  [x] GlobalFoundries Singapore semiconductor wafer inspection
  [x] Detection rate: {detection_rate:.0%} of defects caught automatically
  [x] Grad-CAM enables 83% faster inspector verification (30s -> 5s)
  [x] Projected annual savings: $930K + $2-5M avoided scrap

  KEY INSIGHT: Residual connections and SE attention are not academic
  curiosities -- they are production requirements. ResNets enabled the
  first superhuman image classifiers. SE attention costs almost nothing
  but tells you WHICH features the model cares about. And Grad-CAM
  turns a "black box" classifier into an explainable decision that
  inspectors, regulators, and stakeholders can verify visually.

  Next: In 03_production_pipeline.py, you'll export the best model to
  ONNX and serve it through InferenceServer -- the bridge from
  "it works in a notebook" to "it works in production"...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

